In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Introduction aux primitives

<Accordion>
<AccordionItem title="Versions des packages">

Le code sur cette page a été développé avec les dépendances suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</AccordionItem>
</Accordion>

<span id="qpu-access-patterns"></span>
## Pourquoi Qiskit a-t-il introduit les primitives ?
À l'image des débuts de l'informatique classique, où les développeurs devaient manipuler directement les registres du processeur, la première interface vers les QPU renvoyait simplement les données brutes issues de l'électronique de contrôle.
Ce n'était pas un problème majeur tant que les QPU restaient dans les laboratoires et n'étaient accessibles qu'aux chercheurs.
Conscient que la plupart des développeurs ne devraient pas avoir à transformer ces données brutes en 0 et en 1, Qiskit a introduit `backend.run`, une première abstraction pour accéder aux QPU dans le cloud. Cela a permis aux développeurs
de travailler sur un format de données familier et de se concentrer sur l'essentiel.

À mesure que l'accès aux QPU s'est généralisé et que de nouveaux algorithmes quantiques ont été développés,
le besoin d'une abstraction de plus haut niveau s'est à nouveau fait sentir. En réponse, Qiskit a introduit
l'interface des primitives, optimisée pour deux tâches fondamentales du développement d'algorithmes quantiques :
l'estimation de valeurs d'espérance (`Estimator`) et l'échantillonnage de circuits (`Sampler`). L'objectif est encore une fois
d'aider les développeurs à se concentrer davantage sur l'innovation et moins sur la conversion de données. L'interface des primitives remplace l'interface `backend.run`, puisque `Sampler` offre le même accès direct au matériel que celui proposé par `backend.run`.

## Qu'est-ce qu'une primitive ?
Les systèmes informatiques sont construits sur plusieurs couches d'abstraction. Les abstractions te permettent de te concentrer sur
un niveau de détail particulier, adapté à la tâche en cours. Plus tu te rapproches du matériel,
plus le niveau d'abstraction dont tu as besoin est bas (par exemple, tu pourrais avoir à déplacer ou manipuler des données au niveau des instructions processeur). Plus la tâche que tu veux accomplir est complexe,
plus les abstractions seront de haut niveau (par exemple, tu pourrais utiliser une bibliothèque de programmation pour effectuer
des calculs algébriques).

Dans ce contexte, une *primitive* est la plus petite instruction de traitement, le bloc de construction le plus simple à partir duquel
on peut créer quelque chose d'utile pour un niveau d'abstraction donné.

Les récentes avancées en informatique quantique ont accru le besoin de travailler à des niveaux d'abstraction plus élevés.
À mesure que le domaine évolue vers des unités de traitement quantique (QPU) plus grandes et des flux de travail plus complexes, l'accent se déplace de l'interaction avec les signaux individuels des qubits vers la vision des dispositifs quantiques comme des systèmes qui accomplissent des tâches précises.

Les deux tâches les plus courantes pour les ordinateurs quantiques sont l'échantillonnage d'états quantiques et le calcul de valeurs d'espérance.
Ces tâches ont motivé la conception des primitives Qiskit : **Estimator** et **Sampler**.

- Estimator calcule les valeurs d'espérance d'observables par rapport aux états préparés par des circuits quantiques.
- Sampler échantillonne le registre de sortie à partir de l'exécution de circuits quantiques.

En résumé, le modèle de calcul introduit par les primitives Qiskit rapproche la programmation quantique d'un pas
vers là où se trouve aujourd'hui la programmation classique, où l'accent est moins mis sur les détails matériels et davantage sur les résultats
que tu cherches à obtenir.

## Définition et implémentations des primitives
Il existe deux types de primitives Qiskit : les classes de base et leurs implémentations. Les primitives Estimator et Sampler sont définies par des classes de base primitives open source qui se trouvent dans le SDK Qiskit (dans le module [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives)). Les fournisseurs (tels que Qiskit Runtime) peuvent utiliser ces classes de base pour dériver leurs propres implémentations de Sampler et d'Estimator. La plupart des utilisateurs interagiront avec les implémentations des fournisseurs, et non avec les primitives de base.

### Classes de base
Les primitives `Base` sont des classes abstraites qui définissent une interface commune pour implémenter des primitives. Toutes les autres classes du module [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) héritent de ces classes de base. Les développeurs devraient les utiliser s'ils souhaitent créer leur propre modèle d'exécution basé sur les primitives pour un fournisseur spécifique. Ces classes peuvent également être utiles à ceux qui souhaitent réaliser un traitement très personnalisé et qui trouvent que les implémentations existantes des primitives sont trop simples pour leurs besoins. Les utilisateurs généraux n'utiliseront pas directement les classes de base.

[`BaseEstimatorV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV1) et [`BaseSamplerV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV1) - Bien que les primitives V1 soient toujours utilisables, ces guides se concentrent sur les primitives V2 car elles sont les plus récentes et les plus couramment utilisées.

[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) et [`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) - Les primitives de référence Qiskit suivent ces spécifications d'interface.

<span id="implementations"></span>
### Implémentations
Toutes les primitives sont créées à partir des classes de base ; elles ont donc la même structure générale et le même mode d'utilisation. Par exemple, le format de l'entrée pour toutes les primitives Estimator est identique. Cependant, il existe des différences dans les implémentations qui les rendent uniques.

Voici les implémentations des classes de base des primitives :

- Les [primitives Qiskit Runtime](/guides/qiskit-runtime-primitives), [`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) et [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2), fournissent une implémentation plus sophistiquée (par exemple, en incluant l'atténuation des erreurs) sous forme de service cloud. Cette implémentation des primitives de base est utilisée pour accéder au matériel IBM Quantum&reg;.

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator) et [`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) - Implémentations de référence des primitives qui utilisent le simulateur intégré à Qiskit. Elles sont construites avec le module Qiskit [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information) et produisent des résultats basés sur des simulations de vecteur d'état idéales. Elles sont accessibles via Qiskit. Consulte [Simulation exacte avec les primitives Qiskit](/guides/simulate-with-qiskit-sdk-primitives) pour les détails d'utilisation.

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) et [`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) - Tu peux utiliser ces classes pour « envelopper » n'importe quelle ressource de calcul quantique dans une primitive. Cela te permet d'écrire du code de style primitif pour des fournisseurs qui ne disposent pas encore d'une interface basée sur les primitives. Ces classes peuvent être utilisées exactement comme le Sampler et l'Estimator habituels, sauf qu'elles doivent être initialisées avec un argument `backend` supplémentaire pour sélectionner l'ordinateur quantique à utiliser. Elles sont accessibles via Qiskit. Consulte le guide des [primitives backend](/guides/get-started-with-backend-primitives) pour plus d'informations.

## Options
Tu peux passer des options aux primitives pour les personnaliser selon tes besoins. Bien que l'interface de la méthode `run()` des primitives soit commune à toutes les implémentations, leurs options ne le sont pas. Consulte les références API d'une implémentation de primitive spécifique pour en savoir plus sur les options qu'elle prend en charge.

Par exemple, consulte les rubriques [Options d'Estimator](/guides/estimator-options) et [Options de Sampler](/guides/sampler-options) pour les options des primitives Qiskit Runtime, ou les [références API de Qiskit Aer](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) pour les options des primitives Qiskit Aer.

## Avantages des primitives Qiskit
Avec les primitives, les utilisateurs de Qiskit peuvent écrire du code quantique pour un QPU spécifique sans avoir à gérer explicitement
chaque détail. De plus, grâce à la couche d'abstraction supplémentaire, tu pourrais accéder plus facilement
aux capacités matérielles avancées d'un fournisseur donné. Par exemple, avec les primitives Qiskit Runtime,
tu peux tirer parti des dernières avancées en matière d'atténuation et de suppression des erreurs en activant des options telles que le [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level) de la primitive, plutôt que de construire ta propre implémentation de ces techniques.

Pour les fournisseurs de matériel, implémenter les primitives nativement signifie que tu peux offrir à tes utilisateurs une façon plus « prête à l'emploi »
d'accéder aux fonctionnalités de ton matériel, comme les techniques avancées de post-traitement. Il est ainsi plus facile pour tes utilisateurs de bénéficier des meilleures capacités de ton matériel.

## Étapes suivantes
> **Tip:** - Comprends les [entrées et sorties des primitives](/guides/primitive-input-output).
> - Consulte des [exemples](/guides/simulate-with-qiskit-sdk-primitives) détaillés.
> - Pratique avec les primitives en suivant la [leçon sur les fonctions de coût](/learning/courses/variational-algorithm-design/cost-functions) dans IBM Quantum Learning.
> - Consulte [Créer un fournisseur](/guides/create-a-provider) pour apprendre à implémenter tes propres primitives Sampler et Estimator.
> - Consulte les [références API](https://docs.quantum.ibm.com/api/qiskit/primitives).
> - Lis [Migrer vers les primitives V2](/guides/v2-primitives).
> - Découvre les [primitives Qiskit Runtime](/guides/qiskit-runtime-primitives), utilisées pour exécuter des circuits sur les QPU IBM.